# Native exponential workflow

For all 32 outcomes n, P(n) = exp(-0.1*n)/sum(exp(-0.1*k)). The native program specifies normalized probabilities. Its positive amplitudes and absolute phase are validated against an independent Circuit, separately from readout. No gate decomposition is written here.

Run every cell from the repository root. See WORKFLOW.md for acceptance and scope.

This prepares a fresh output using prepare_state; it does not preserve the original exponential routine's in-place unitary on arbitrary nonzero inputs.


In [ ]:
import math
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QArray, QBit, prepare_state, qfunc

weights = [math.exp(-0.1 * value) for value in range(32)]
probabilities = [value / math.fsum(weights) for value in weights]

@qfunc
def main(probabilities: list[float], x: Output[QArray[QBit]]) -> None:
    prepare_state(probabilities, 0, x)


In [ ]:
program = synth.synthesize(main, classical_inputs={"probabilities": probabilities}, max_workspace=0)
result = fq.run(program)
probabilities = dict(program.probabilities(result, registers=('x',)))
print(probabilities)
print(program.report["resources"])


In [ ]:
expected_weights = [math.exp(-0.1 * value) for value in range(32)]
normalizer = math.fsum(expected_weights)
assert set(probabilities) == {(value,) for value in range(32)}
assert all(abs(probabilities[(value,)] - weight / normalizer) < 1e-12 for value, weight in enumerate(expected_weights))
